# Synthetic Ramesside Star Clocks

In [1]:
import numpy as np
import sys 
from pathlib import Path
import pandas as pd

# Set up paths
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))
#import decanopy

# I want get_sun to shut up so ignoring warnings:
import warnings
warnings.simplefilter('ignore', UserWarning)


## Import Sky & Initialize 

Due to ongoing code cleaning, the pathing for the inputs and outputs is still being reworked a little. For now, specify one of two runs manually: 
- Option 1: "real sky", aka Hipparcos data for 1300 BCE. 
- Option 2: "rand sky", aka one specfic procedurally generated sky with 518 stars

The data is organized as follows. In each case,`[type]` = `real_sky` or `rand_sky`. 
- The night sky data input files (star names, Dec/RAs, & mags) are stored in `/decanOpy/data/input/skyflow/[type]`.
- The night sky data output ("skyflow") outputs are stored in `/decanOpy/data/output/skyflow/[type]`.
- The synthetic RSCs are stored in `/decanOpy/data/output/rsc/[type]`. 

### Option 1: Run this for real_sky

In [2]:
from decanopy.config import DEFAULT_STAR_DATA, SKYFLOW_OUTPUT_REAL_SKY, RSC_OUTPUT_REAL_SKY, DEFAULT_STAR_NAMES
filename = "hippdata1300BC.txt"
filepath = SKYFLOW_OUTPUT_REAL_SKY / filename 

# make dictionary of names and magnitudes (if not already in memory) 
name_df = pd.read_csv(DEFAULT_STAR_NAMES, index_col=None, header=0, names=['Name', 'RA', 'Dec', 'Mag'])
mag_dict = name_df.set_index('Name')['Mag'].to_dict()

# determine output writepath
writepath = RSC_OUTPUT_REAL_SKY 

### Option 2: Run this for rand_sky

In [3]:
from decanopy.config import SKYFLOW_OUTPUT_RAND_SKY, USER_INPUT_RAND_SKY, RSC_OUTPUT_RAND_SKY
filename = 'mockdata_518_1300BC-Mar-18-2024_1059.txt'
filepath = SKYFLOW_OUTPUT_RAND_SKY / filename 

# make dictionary of names and magnitudes (if not already in memory) 
ic_filename = 'star_data_Mar-18-2024_1059.csv'
name_df = pd.read_csv(USER_INPUT_RAND_SKY / ic_filename, index_col=None, header=0, names=['Name', 'RA', 'Dec', 'Mag'])
mag_dict = name_df.set_index('Name')['Mag'].to_dict()

# determine output writepath
writepath = RSC_OUTPUT_RAND_SKY 

### Common: setup structures

In [4]:
### Read in data and prep structures
## (I promise I'll make this into a nicer initialization functon,
##  but just run it as is for now. )
## takes about 40 seconds on my computer 

from decanopy.models.RSC.syn_rsc import StarRiseSet, isStarVisible

# get data
decanOutput = pd.read_csv(filepath, sep = "|")
# get header 
header = decanOutput.keys()

# get list of star names
starlist = []
for star_name in header[4:-1:2]:
    starlist.append(star_name[0:-8])
# get standard data 
jd = decanOutput[header[0]].to_numpy()
hrd = decanOutput[header[1]]
sunAz = decanOutput[header[2]].to_numpy()
sunAlt = decanOutput[header[3]].to_numpy()

# get star data
num_decs = int((len(header) - 4) / 2) # how many stars in list 
starsAz = np.zeros((num_decs, len(jd)))
starsAlt = np.zeros((num_decs, len(jd)))
for i in range(0, num_decs):
    starsAz[i, :] = decanOutput[header[4 + 2 * i]].to_numpy()
    starsAlt[i, :] = decanOutput[header[5 + 2 * i]].to_numpy()
   
# get sunrise and sunset times
(sunRise, sunSet) = StarRiseSet(jd, sunAlt, -12)
sunAzSet = sunAz[sunSet]

# get star rise and set times
rise_alt_deg = 10 # define starrise in muber of degrees above horizon 
starAzRiseList = np.zeros((num_decs, len(sunRise)))
starVisList = np.full((num_decs, len(sunRise)), True)
starMaxAltList = np.zeros((num_decs, len(sunRise)-1))
# loop over all decans
for i in range(0, num_decs):
    # create visibility and maximum altitude array
    if np.min(starsAlt[i]) >= rise_alt_deg: 
        # if the star is circumpolar at the given location
        starVisList[i,:] = np.full((365,), True)
        starMaxAltList[i,:] = np.max(starsAlt[i])
    elif np.max(starsAlt[i]) < rise_alt_deg:
        # if a star never rises above rise_alt_deg
        starVisList[i,:] = np.full((365,), False)
        starMaxAltList[i,:] = np.max(starsAlt[i])
    else: 
        # otherwise, check when the star isn't visible
        (starRise, starSet) = StarRiseSet(jd, starsAlt[i], rise_alt_deg)    
        starAzRise = starsAz[i, starRise]                                   
        starAzRiseList[i, :] = starAzRise
        (maxAlt, starVis) = isStarVisible(sunSet, sunRise, starsAlt[i])
        starVisList[i,:] = starVis
        starMaxAltList[i,:] = maxAlt

## Create Synthetic RSCs

In [5]:
# Define parameters for synRSC model 
alt_window = (0, 45) # degrees 
horizon = (180-7, 180 + 7) # Note: MUST have smaller number first; 
bsize = 1 # bin size (must be 1 if gsize = 0)
gsize = 0 # gap size (relative to binsize) 

# Name output file
writename = filename[0:-4] + str(horizon[0]) + '-' + str(horizon[1]) + '_b=' + str(bsize) + '_g=' + str(gsize) + '_' + str(num_decs) + '.xlsx'
# writename = 'something_else' + '.xlsx' ## uncomment and edit for user-specified filename

In [7]:
## (I promise this willa also be turned into a nice write function,
##  but just run it as is for now. )

from decanopy.models.RSC.syn_rsc import synRSC, mag_data, name_or_mag_data, dbc_data, full_choice_data

# Creating Excel Writer Object from Pandas  
writer = pd.ExcelWriter(writepath / writename, engine='xlsxwriter')     
workbook=writer.book
worksheet=workbook.add_worksheet('RSCs')
writer.sheets['RSCs'] = worksheet
worksheet2=workbook.add_worksheet('Mag Select')
writer.sheets['Mag Select'] = worksheet2
worksheet3=workbook.add_worksheet('Name Select')
writer.sheets['Name Select'] = worksheet3
worksheet4=workbook.add_worksheet('DBC Select')
writer.sheets['DBC Select'] = worksheet4
worksheet5=workbook.add_worksheet('Full Choice')
writer.sheets['Full Choice'] = worksheet5

# Format test
format = workbook.add_format()
format.set_font_size(11)

# Write down some important data
worksheet.write(0, 0, "horizon is " + str(horizon), format)
worksheet.write(1, 0, "alt window is " + str(alt_window), format)
worksheet.write(2, 0, "bsize = " + str(bsize), format)
worksheet.write(3, 0, "gsize = " + str(gsize), format)

# Write data (saved in /SynRsc folder) # UPDATE FOLDER STRUCTURE
known_stars_dict = {}
all_choices_dict = {}
dbc_dict = {i: {} for i in range(24)}  # initialize dbc dict for each table

for i in range(0, 24):
    date = i * 15 # days from first day in decan data
    df, dbc_table = synRSC(date, alt_window, horizon, bsize, gsize, sunSet, sunRise, starlist, starsAz, starsAlt, starVisList)
    dbc_dict[i] = dbc_table # store dbc_table for each table date
    df.to_excel(writer, sheet_name='RSCs',startrow= i * 15 + 5, startcol=0)   
    worksheet.write(i * 15 + 5,  0, "Table " + str(i + 1), format)
    #add mag data
    df_mag = mag_data(df, mag_dict)
    df_mag.to_excel(writer, sheet_name='Mag Select',startrow= i * 15 + 5, startcol=0) 
    worksheet2.write(i * 15 + 5,  0, "Table " + str(i + 1), format)
    # add name or mag data
    (df_name, known_stars_dict) = name_or_mag_data(df, mag_dict, known_stars_dict)
    df_name.to_excel(writer, sheet_name='Name Select',startrow= i * 15 + 5, startcol=0) 
    worksheet3.write(i * 15 + 5,  0, "Table " + str(i + 1), format)
    worksheet3.write(4,  10, "Number of known stars = " + str(len((known_stars_dict))), format)
    df_dict3 = pd.DataFrame(list(known_stars_dict.items()), columns=["H-index", "'Known' index"])
    df_dict3.to_excel(writer, sheet_name='Name Select', startrow=5, startcol=10, index=False)
    # add dbc data 
    df_dbc = dbc_data(df, dbc_dict[i])
    df_dbc.to_excel(writer, sheet_name='DBC Select', startrow= i * 15 + 5, startcol=0) 
    worksheet4.write(i * 15 + 5,  0, "Table " + str(i + 1), format)
    # add choices data 
    df_choices, choices_dict  = full_choice_data(df, mag_dict, dbc_table)
    df_choices.to_excel(writer, sheet_name='Full Choice', startrow= i * 15 + 5, startcol=0) 
    worksheet5.write(i * 15 + 5,  0, "Table " + str(i + 1), format)
    # update all_choices_dict
    for k, v in choices_dict.items():
        all_choices_dict[k] = all_choices_dict.get(k, 0) + v
# write final all_choices_dict and close    
df_dict5 = pd.DataFrame(list(all_choices_dict.items()), columns=["Code", "Count"])
df_dict5.to_excel(writer, sheet_name='Full Choice', startrow=5, startcol=12, index=False)    
writer.close()